# Round 3 Data Exploration

Products: **HYDROGEL_PACK**, **VELVETFRUIT_EXTRACT**, and VEV_xxxx strikes (likely options on VELVETFRUIT_EXTRACT)  
Days: 0, 1, 2

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotter import Plotter

DATA_DIR = "../data/round3"
days = [0, 1, 2]

In [ ]:
all_prices = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/prices_round_3_day_{d}.csv", delimiter=";") for d in days],
    ignore_index=True,
)
all_prices['spread'] = all_prices['ask_price_1'] - all_prices['bid_price_1']

products = sorted(all_prices['product'].unique())
print(f"total rows: {len(all_prices)}")
for p in products:
    print(f"  {p}: {(all_prices['product'] == p).sum()} rows")

In [ ]:
all_trades = pd.concat(
    [pd.read_csv(f"{DATA_DIR}/trades_round_3_day_{d}.csv", delimiter=";").assign(day=d) for d in days],
    ignore_index=True,
)
all_trades = all_trades.rename(columns={"symbol": "product"})
all_trades = all_trades.merge(all_prices, on=["day", "timestamp", "product"], how="left")

print(f"total trades: {len(all_trades)}")
for p in sorted(all_trades['product'].unique()):
    print(f"  {p}: {(all_trades['product'] == p).sum()} trades")

In [ ]:
prices_by_prod = {p: all_prices[all_prices['product'] == p].reset_index(drop=True) for p in products}
trades_by_prod = {p: all_trades[all_trades['product'] == p].reset_index(drop=True) for p in products}

## Mid Price Across Days (all products)

In [ ]:
DAY_OFFSET = 1_000_000
all_prices['global_ts'] = all_prices['timestamp'] + (all_prices['day'] - all_prices['day'].min()) * DAY_OFFSET

fig = go.Figure()
for p in products:
    df = all_prices[all_prices['product'] == p].sort_values('global_ts')
    fig.add_trace(go.Scattergl(x=df['global_ts'], y=df['mid_price'], mode='lines', name=p))
fig.update_layout(title='Mid price — all products, all days', xaxis_title='global timestamp', yaxis_title='mid price', height=600)
fig.show(renderer='browser')

## Spread Summary Per Product

In [ ]:
rows = []
for p in products:
    s = prices_by_prod[p]['spread'].dropna()
    rows.append({'product': p, 'mean': s.mean(), 'median': s.median(), 'min': s.min(), 'max': s.max(), 'n': len(s)})
pd.DataFrame(rows)

## VEV Strikes vs Underlying

If VEV_xxxx are options with strike xxxx, their mids should track intrinsic value of VELVETFRUIT_EXTRACT.

In [ ]:
vev_products = [p for p in products if p.startswith('VEV_')]
underlying = 'VELVETFRUIT_EXTRACT'

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5],
                    subplot_titles=('Underlying mid (VELVETFRUIT_EXTRACT)', 'VEV strike mids'))

u = all_prices[all_prices['product'] == underlying].sort_values('global_ts')
fig.add_trace(go.Scattergl(x=u['global_ts'], y=u['mid_price'], mode='lines', name=underlying), row=1, col=1)

for p in vev_products:
    df = all_prices[all_prices['product'] == p].sort_values('global_ts')
    fig.add_trace(go.Scattergl(x=df['global_ts'], y=df['mid_price'], mode='lines', name=p), row=2, col=1)

fig.update_layout(title=f'{underlying} vs VEV strikes', height=700)
fig.show(renderer='browser')

## Order Book Viewer

In [ ]:
p = Plotter(
    [f"{DATA_DIR}/prices_round_3_day_{d}.csv" for d in days],
    [f"{DATA_DIR}/trades_round_3_day_{d}.csv" for d in days],
)
p.visualize_orderbook(product="VELVETFRUIT_EXTRACT", renderer="browser")

In [ ]:
p.visualize_orderbook(product="HYDROGEL_PACK", renderer="browser")